In [23]:
import os
import json
import warnings
import librosa
import numpy as np
import yt_dlp
from openai import OpenAI
import speech_recognition as sr
from moviepy.editor import AudioFileClip
import random

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

class ToneAnalyzer:
    def __init__(self, audio_path):
        self.audio_path = audio_path
        self.y = None
        self.sr = 22050
        
    def analyze(self):
        print(f"[INFO] Analyzing vocal tone...")
        temp_wav = "temp_tone.wav"
        
        try:
            # Step 1: Use MoviePy to convert to a standard WAV file
            # This bypasses the "arrays to stack" error by avoiding direct memory access
            with AudioFileClip(self.audio_path) as clip:
                # Analyze first 60 seconds
                subclip = clip.subclip(0, min(60, clip.duration))
                subclip.write_audiofile(temp_wav, codec='pcm_s16le', verbose=False, logger=None)
            
            # Step 2: Load the clean WAV with Librosa (No FFmpeg needed for WAV)
            self.y, self.sr = librosa.load(temp_wav, sr=None)
            
            # A. Energy (Volume/Confidence)
            rms = librosa.feature.rms(y=self.y)
            energy = min(100, np.mean(rms) * 1000)
            
            # B. Pitch (Expressiveness)
            pitches, _ = librosa.piptrack(y=self.y, sr=self.sr)
            pitch_vals = pitches[pitches > 0]
            expressiveness = min(100, (np.std(pitch_vals) / 50) * 100) if len(pitch_vals) > 0 else 0

            # Cleanup
            if os.path.exists(temp_wav): os.remove(temp_wav)

            return {
                "energy": round(energy, 2), 
                "expressiveness": round(expressiveness, 2)
            }
        except Exception as e:
            print(f"[WARN] Tone analysis skipped: {e}")
            if os.path.exists(temp_wav): os.remove(temp_wav)
            return {"energy": 0, "expressiveness": 0}

In [24]:
class FreeTranscriber:
    def transcribe(self, audio_path):
        print("[INFO] Transcribing (Free Mode)...")
        recognizer = sr.Recognizer()
        
        # We need a clean WAV file for SpeechRecognition
        wav_path = "temp_clean.wav"
        try:
            # 1. Convert to WAV using MoviePy (Internal FFMPEG)
            with AudioFileClip(audio_path) as clip:
                # Limit to 2 minutes to keep it fast/free
                subclip = clip.subclip(0, min(120, clip.duration))
                subclip.write_audiofile(wav_path, codec='pcm_s16le', verbose=False, logger=None)
            
            # 2. Send to Google Free API
            with sr.AudioFile(wav_path) as source:
                audio_data = recognizer.record(source)
                text = recognizer.recognize_google(audio_data)
                
            # Cleanup
            if os.path.exists(wav_path): os.remove(wav_path)
            return text
            
        except sr.UnknownValueError:
            return "(Unintelligible speech)"
        except Exception as e:
            print(f"[ERROR] Transcription failed: {e}")
            return ""

In [ ]:
class RuleBasedSharks:
    def deliberate(self, transcript, tone_data):
        print("[INFO] Sharks are deliberating (Logic Engine)...")
        text = transcript.lower()
        
        # --- 1. KEYWORD LISTS ---
        # Expanded lists to catch more variations
        kw_sales = ['sales', 'sold', 'revenue', 'dollars', 'profit', 'orders', 'cash', 'making money', 'income']
        kw_patents = ['patent', 'proprietary', 'unique', 'invented', 'utility', 'design', 'intellectual property']
        kw_margins = ['margin', 'cost', 'make it for', 'sell it for', 'retail', 'wholesale', 'price', 'production']
        kw_customer = ['problem', 'solve', 'customer', 'user', 'pain', 'need', 'help', 'people', 'everyone']
        
        # --- 2. DETECTION ---
        has_sales = any(w in text for w in kw_sales)
        has_patents = any(w in text for w in kw_patents)
        has_margins = any(w in text for w in kw_margins)
        has_customer_focus = any(w in text for w in kw_customer)
        has_energy = tone_data['energy'] > 20
        
        # --- 3. DEBUG PRINTS (See what is happening!) ---
        print(f"   [DEBUG] Found Sales Keywords?    {has_sales}")
        print(f"   [DEBUG] Found Margin Keywords?   {has_margins}")
        print(f"   [DEBUG] Found Patent Keywords?   {has_patents}")
        print(f"   [DEBUG] Found Customer Keywords? {has_customer_focus}")
        print(f"   [DEBUG] High Energy?             {has_energy}")

        # --- 4. SCORING ---
        score = 40 # Base
        if has_sales: score += 15
        if has_margins: score += 15
        if has_patents: score += 10
        if has_customer_focus: score += 10
        if has_energy: score += 10
        
        # --- 5. FEEDBACK GENERATION ---
        feedback = {}
        
        # A. The Finance Shark
        finance_invest = ["INVEST. I smell money.", "INVEST. Good margins.", "INVEST. I like the sales numbers."]
        finance_reject = ["NO DEAL. No sales mentioned?", "NO DEAL. I don't know your margins.", "NO DEAL. No revenue, no deal."]
        
        if has_margins and has_sales:
            feedback["The Finance Shark"] = random.choice(finance_invest)
        else:
            feedback["The Finance Shark"] = random.choice(finance_reject)
            
        # B. The Visionary
        visionary_invest = ["INVEST. Great vision.", "INVEST. This is a hero product.", "INVEST. I love the innovation."]
        visionary_reject = ["NO DEAL. Nothing proprietary.", "NO DEAL. It's just a product, not a company.", "NO DEAL. I'm out."]
        
        if has_patents or tone_data['expressiveness'] > 40:
            feedback["The Visionary"] = random.choice(visionary_invest)
        else:
            feedback["The Visionary"] = random.choice(visionary_reject)

        # C. The Customer Advocate
        advocate_invest = ["INVEST. You solve a real problem.", "INVEST. Clear and simple.", "INVEST. Consumers will love this."]
        advocate_reject = ["NO DEAL. Who is this for?", "NO DEAL. Too confusing.", "NO DEAL. What problem are you solving?"]

        if has_customer_focus:
            feedback["The Customer Advocate"] = random.choice(advocate_invest)
        else:
            feedback["The Customer Advocate"] = random.choice(advocate_reject)
            
        # D. The Skeptic
        skeptic_critiques = ["NO DEAL. Too much competition.", "NO DEAL. Too early.", "NO DEAL. Valuation is crazy."]
        feedback["The Skeptic"] = random.choice(skeptic_critiques)
        
        return feedback, score

In [26]:
def download_audio(url):
    print(f"[INFO] Downloading {url}...")
    base = "temp_pitch"
    # Clean old files
    for ext in [".mp3", ".m4a", ".wav"]:
        if os.path.exists(base + ext): os.remove(base + ext)

    ydl_opts = {
        'format': 'bestaudio[ext=m4a]/bestaudio/best',
        'outtmpl': base + '.%(ext)s',
        'quiet': True, 'no_warnings': True
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            return f"{base}.{info['ext']}"
    except Exception as e:
        print(f"[ERROR] Download failed: {e}")
        return None

In [28]:
if __name__ == "__main__":
    # Test URL: Scrub Daddy Pitch
    URL ="https://www.youtube.com/watch?v=ae5MssJ8en4"
    print("--- STARTING FREE SHARK ANALYZER ---")
    
    # 1. Download
    audio_path = download_audio(URL)
    
    if audio_path:
        # 2. Tone Analysis
        tone_engine = ToneAnalyzer(audio_path)
        tone_metrics = tone_engine.analyze()
        print(f"   > Energy: {tone_metrics['energy']} | Expressiveness: {tone_metrics['expressiveness']}")
        
        # 3. Transcription
        transcriber = FreeTranscriber()
        transcript = transcriber.transcribe(audio_path)
        print(f"   > Transcript Snippet: \"{transcript[:100]}...\"")
        
        if transcript:
            # 4. Shark Verdicts
            sharks, score = RuleBasedSharks().deliberate(transcript, tone_metrics)
            
            print("\n" + "="*40)
            print("        FINAL VERDICT")
            print("="*40)
            print(f" Business Score: {score}/100")
            for shark, verdict in sharks.items():
                print(f"\n🔹 {shark}:\n   \"{verdict}\"")
        
        # Cleanup
        try:
            if os.path.exists(audio_path): os.remove(audio_path)
        except: pass

--- STARTING FREE SHARK ANALYZER ---
[INFO] Downloading https://www.youtube.com/watch?v=ae5MssJ8en4...
[INFO] Analyzing vocal tone...                             
   > Energy: 44.619998931884766 | Expressiveness: 100
[INFO] Transcribing (Free Mode)...
   > Transcript Snippet: "high charge in the world today temperature..."
[INFO] Sharks are deliberating (Logic Engine)...

        FINAL VERDICT
 Business Score: 50/100

🔹 The Finance Shark:
   "NO DEAL. You didn't mention your margins or revenue. I'm out."

🔹 The Visionary:
   "INVEST. I see the vision and the energy. This could be huge."

🔹 The Customer Advocate:
   "NO DEAL. I'm confused. Who is this for? You never explained the actual problem."

🔹 The Skeptic:
   "NO DEAL. I just don't get it. I'm out."
